# Network Port Scanner and Threats Visualizer
**Author:** Suhani Bhosale

## Objective
Scan a limited list of TCP ports on localhost or a private IPv4 host, identify open services, attach simple rules-based review notes, and visualize the results.

**Safety:** Only scan systems you own or have explicit permission to assess. The code rejects public IPv4 addresses and hostnames. Open ports are not proof of vulnerabilities.

## 1. Install dependencies
Run this once in your notebook environment if the packages are missing: `pip install -r requirements.txt`

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image


## 2. Scanner implementation

In [ ]:
"""
Network Port Scanner and Threats Visualizer
Author: Suhani Bhosale

Safety: This educational tool is deliberately limited to localhost and private IPv4
addresses. Scan only systems you own or have explicit permission to assess.
"""
from __future__ import annotations

import argparse
import ipaddress
import socket
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable

import pandas as pd
import matplotlib.pyplot as plt

COMMON_PORTS = [21, 22, 23, 25, 53, 80, 110, 139, 143, 443, 445, 3306, 3389, 5432, 8080]
SERVICE_NAMES = {
    21: "FTP", 22: "SSH", 23: "Telnet", 25: "SMTP", 53: "DNS",
    80: "HTTP", 110: "POP3", 139: "NetBIOS", 143: "IMAP",
    443: "HTTPS", 445: "SMB", 3306: "MySQL", 3389: "RDP",
    5432: "PostgreSQL", 8080: "HTTP-alt"
}
RISK_GUIDANCE = {
    21: ("High", "FTP may transmit credentials without encryption; prefer SFTP/FTPS."),
    22: ("Medium", "SSH is common for administration; restrict access and use keys."),
    23: ("High", "Telnet is unencrypted; disable it and use SSH."),
    25: ("Medium", "SMTP should be configured securely and not act as an open relay."),
    53: ("Medium", "Check DNS exposure and restrict recursion where appropriate."),
    80: ("Low", "HTTP is unencrypted; redirect sensitive web traffic to HTTPS."),
    110: ("High", "POP3 may be unencrypted; prefer POP3S or disable if unused."),
    139: ("High", "Legacy NetBIOS exposure may reveal network information; restrict it."),
    143: ("Medium", "Use encrypted IMAP (IMAPS) and restrict access."),
    443: ("Low", "HTTPS is expected; keep TLS configuration and software updated."),
    445: ("High", "SMB exposure can be risky; block from untrusted networks and patch."),
    3306: ("High", "Avoid exposing MySQL publicly; bind to trusted interfaces."),
    3389: ("High", "RDP is a frequent target; restrict with VPN/firewall and MFA."),
    5432: ("High", "Avoid exposing PostgreSQL publicly; restrict trusted clients."),
    8080: ("Medium", "Review alternate web service access controls and authentication.")
}

def validate_target(target: str) -> str:
    """Allow only loopback or RFC1918 IPv4 targets; no public Internet scans."""
    try:
        ip = ipaddress.ip_address(target.strip())
    except ValueError as exc:
        raise ValueError("Enter an IPv4 address (hostname input is not enabled).") from exc
    if ip.version != 4:
        raise ValueError("Only IPv4 targets are supported in this demo.")
    if not (ip.is_loopback or ip.is_private):
        raise ValueError("For safety, this project scans only localhost/private IPv4 targets.")
    return str(ip)

def scan_tcp_ports(target: str, ports: Iterable[int], timeout: float = 0.35) -> pd.DataFrame:
    """Perform a basic TCP connect scan and return results as a DataFrame."""
    target = validate_target(target)
    rows = []
    for port in ports:
        if not isinstance(port, int) or not 1 <= port <= 65535:
            raise ValueError(f"Invalid TCP port: {port!r}")
        started = time.perf_counter()
        state = "closed_or_filtered"
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
                sock.settimeout(timeout)
                result = sock.connect_ex((target, port))
                state = "open" if result == 0 else "closed_or_filtered"
        except (OSError, TimeoutError):
            state = "closed_or_filtered"
        elapsed_ms = round((time.perf_counter() - started) * 1000, 2)
        risk, guidance = RISK_GUIDANCE.get(port, ("Review", "Identify the service and restrict access if not required."))
        rows.append({
            "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "target": target,
            "port": port,
            "service_guess": SERVICE_NAMES.get(port, "Unknown"),
            "state": state,
            "risk_if_open": risk if state == "open" else "N/A",
            "security_note": guidance if state == "open" else "No TCP listener detected from this scan.",
            "response_ms": elapsed_ms
        })
    return pd.DataFrame(rows)

def add_threat_labels(results: pd.DataFrame) -> pd.DataFrame:
    """Add a simple rules-based priority label; not a vulnerability verdict."""
    results = results.copy()
    def label(row):
        if row["state"] != "open":
            return "No open service detected"
        if row["risk_if_open"] == "High":
            return "Priority review"
        if row["risk_if_open"] == "Medium":
            return "Review configuration"
        return "Routine review"
    results["threat_label"] = results.apply(label, axis=1)
    return results

def visualize_results(results: pd.DataFrame, output_dir: str = "output") -> tuple[Path, Path]:
    """Save CSV results and a port-state/risk visualization."""
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    csv_path = out / "scan_results.csv"
    image_path = out / "threats_visualization.png"
    results.to_csv(csv_path, index=False)

    open_rows = results[results["state"] == "open"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    state_counts = results["state"].value_counts()
    axes[0].bar(state_counts.index, state_counts.values)
    axes[0].set_title("TCP Port Scan Results")
    axes[0].set_ylabel("Number of ports")
    axes[0].tick_params(axis="x", rotation=15)

    if open_rows.empty:
        axes[1].text(0.5, 0.5, "No open ports detected", ha="center", va="center", fontsize=12)
        axes[1].set_xticks([])
        axes[1].set_yticks([])
        axes[1].set_title("Rules-based Review Priority")
    else:
        order = ["High", "Medium", "Low", "Review"]
        counts = open_rows["risk_if_open"].value_counts().reindex(order, fill_value=0)
        axes[1].bar(counts.index, counts.values)
        axes[1].set_title("Open Ports by Review Priority")
        axes[1].set_ylabel("Number of open ports")
    fig.suptitle("Network Port Scanner & Threats Visualizer")
    fig.tight_layout()
    fig.savefig(image_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return csv_path, image_path

def run_scan(target: str, ports: Iterable[int] = COMMON_PORTS, timeout: float = 0.35,
             output_dir: str = "output") -> pd.DataFrame:
    results = add_threat_labels(scan_tcp_ports(target, ports, timeout))
    csv_path, image_path = visualize_results(results, output_dir)
    print(f"Target: {validate_target(target)}")
    print(f"Ports checked: {len(results)} | Open: {(results['state'] == 'open').sum()}")
    print(f"CSV saved: {csv_path.resolve()}")
    print(f"Visualization saved: {image_path.resolve()}")
    return results

def main():
    parser = argparse.ArgumentParser(description="Local/private IPv4 TCP port scanner and simple threat visualizer.")
    parser.add_argument("--target", default="127.0.0.1", help="localhost or private IPv4 address only (default: 127.0.0.1)")
    parser.add_argument("--ports", default=",".join(map(str, COMMON_PORTS)), help="Comma-separated TCP ports, e.g. 22,80,443")
    parser.add_argument("--timeout", type=float, default=0.35, help="Per-port timeout in seconds (default: 0.35)")
    parser.add_argument("--output", default="output", help="Output directory")
    args = parser.parse_args()
    try:
        ports = [int(item.strip()) for item in args.ports.split(",") if item.strip()]
        if not 0 < args.timeout <= 5:
            raise ValueError("Timeout must be greater than 0 and no more than 5 seconds.")
        results = run_scan(args.target, ports, args.timeout, args.output)
        print(results[["port", "service_guess", "state", "risk_if_open", "threat_label"]].to_string(index=False))
    except (ValueError, OSError) as exc:
        parser.error(str(exc))



## 3. Run a localhost demo
The demo checks a small set of ports on `127.0.0.1`. Results depend on which services are running on your own machine.

In [ ]:
demo_ports = [22, 80, 443, 8080]
demo_results = run_scan("127.0.0.1", demo_ports, timeout=0.35, output_dir="output")
display(demo_results[["port", "service_guess", "state", "risk_if_open", "threat_label"]])
display(Image(filename="output/threats_visualization.png"))

## 4. Interpret the results
- **open:** a TCP connection succeeded during the scan.
- **closed_or_filtered:** the connection did not succeed; this may indicate a closed port, firewall filtering, or a timeout.
- **Priority review / Review configuration / Routine review:** generic rule-based prompts for services detected open. They are not vulnerability findings.

## References
- [Python socket documentation](https://docs.python.org/3/library/socket.html)
- [Nmap port-scanning overview](https://nmap.org/book/port-scanning.html)
- [Nmap legal and authorization guidance](https://nmap.org/book/legal-issues.html)